<a href="https://colab.research.google.com/github/praneetha-glitch/BIS-Lab/blob/main/1BF24CS163_WEEK2_PSO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import cv2


np.random.seed(42)
dummy_image = np.zeros((200, 200), dtype=np.uint8)
dummy_image[10:100, 10:100] = 180
dummy_image = cv2.GaussianBlur(dummy_image, (5, 5), 0)


HISTOGRAM, _ = np.histogram(dummy_image, bins=256, range=(0, 256))
TOTAL_PIXELS = dummy_image.size

def calculate_otsu_variance(threshold):
    """
    Calculates Otsu's Between-Class Variance for a given threshold value.
    Higher variance indicates superior background/foreground separation contrast.
    """
    t = int(np.clip(round(threshold), 0, 255))


    w0 = np.sum(HISTOGRAM[:t]) / TOTAL_PIXELS
    w1 = np.sum(HISTOGRAM[t:]) / TOTAL_PIXELS


    if w0 == 0 or w1 == 0:
        return 0.0


    idx_0 = np.arange(0, t)
    idx_1 = np.arange(t, 256)

    mu0 = np.sum(idx_0 * HISTOGRAM[:t]) / (w0 * TOTAL_PIXELS)
    mu1 = np.sum(idx_1 * HISTOGRAM[t:]) / (w1 * TOTAL_PIXELS)


    return w0 * w1 * ((mu0 - mu1) ** 2)


NUM_PARTICLES = 15
MAX_ITERATIONS = 20
W = 0.5
C1 = 1.5
C2 = 1.5


X_MIN, X_MAX = 0.0, 255.0
V_MIN, V_MAX = -20.0, 20.0


positions = np.random.uniform(X_MIN, X_MAX, NUM_PARTICLES)
velocities = np.random.uniform(V_MIN, V_MAX, NUM_PARTICLES)


p_best_positions = np.copy(positions)
p_best_fitness = np.array([calculate_otsu_variance(p) for p in positions])

g_best_idx = np.argmax(p_best_fitness)
g_best_position = p_best_positions[g_best_idx]
g_best_fitness = p_best_fitness[g_best_idx]


for iteration in range(MAX_ITERATIONS):
    for i in range(NUM_PARTICLES):
        r1, r2 = np.random.rand(), np.random.rand()


        cognitive = C1 * r1 * (p_best_positions[i] - positions[i])
        social = C2 * r2 * (g_best_position - positions[i])
        velocities[i] = (W * velocities[i]) + cognitive + social
        velocities[i] = np.clip(velocities[i], V_MIN, V_MAX)


        positions[i] = positions[i] + velocities[i]
        positions[i] = np.clip(positions[i], X_MIN, X_MAX) # Boundary Check


        current_fitness = calculate_otsu_variance(positions[i])

        if current_fitness > p_best_fitness[i]:
            p_best_fitness[i] = current_fitness
            p_best_positions[i] = positions[i]

            if current_fitness > g_best_fitness:
                g_best_fitness = current_fitness
                g_best_position = positions[i]


optimal_threshold = int(np.round(g_best_position))
print(f"Optimal Threshold Discovered by PSO: {optimal_threshold}")
print(f"Maximized Between-Class Variance: {g_best_fitness:.4f}")


_, segmented_img = cv2.threshold(dummy_image, optimal_threshold, 255, cv2.THRESH_BINARY)

print(f"Unique pixel values in segmented output: {np.unique(segmented_img)}")


Optimal Threshold Discovered by PSO: 96
Maximized Between-Class Variance: 5018.4891
Unique pixel values in segmented output: [  0 255]
